# COSI full legacy core3 old-LD AD7 × eQTL18 rerun

Human-operated notebook following the xQTL-runner visible submission pattern. Every retained workflow submission is a complete `bsub <<'LSF'` cell with its runtime activation and literal Snakemake command visible.

Execution chain: `operator notebook → YAML → bsub → Snakemake → local COSI Snakefile → SoS → canonical implementation modules`. No wrapper script, secondary controller, or hidden command generator is used.

In [ ]:
%%bash
set -eo pipefail
ROOT=/sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun
MUNGER_ROOT=/sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger
sha256sum \
  "$ROOT/config/full_legacy_core3_oldld_ad7_eqtl18.yaml" \
  "$ROOT/config/regions/GRCh38_103.legacy_minmax.noAPOE_noHLA.18751.tsv" \
  "$ROOT/workflow/snakefile_combined.py" \
  /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/1_DataPrep/Snakemake_38/config_mesa_mono_v5_white_pve70_sensitivity_hg38.yaml \
  /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/1_DataPrep/Snakemake_38/Snakefile
/sc/arion/work/sunh14/conda/envs/cosi/bin/python - <<'PY'
from pathlib import Path
import yaml
root = Path('/sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun')
with (root / 'config/full_legacy_core3_oldld_ad7_eqtl18.yaml').open() as handle:
    cfg = yaml.safe_load(handle)
genes = sum(1 for _ in (root / 'config/regions/GRCh38_103.legacy_minmax.noAPOE_noHLA.18751.tsv').open()) - 1
assert len(cfg['eqtl']) == 18
assert len(cfg['gwas']) == 7
assert cfg['expected_pair_count'] == 126
assert genes == 18751
assert not {'AB42', 'pTau181', 'tTau'} & set(cfg['gwas'])
assert cfg['eqtl']['MESA_mono_v5_white']['file'].endswith('mesa_mono_v5_white.pve70_sensitivity.TensorQTL.cis_qtl_pairs.chrall.CPRA_b38.vcf.gz')
print('contract=PASS eqtl=18 gwas=7 pairs=126 genes=18751')
PY

## 2. MESA PVE70 munger production and resume

Submit this cell when the declared MESA products are absent. Rerun the same cell without force to resume. Historical production `268569158` failed at 22/178 because its child jobs inherited `py3.11` and could not find `vt` or `bcftools`; the command below fixes that launch defect.

In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger/logs
bsub <<'LSF'
#BSUB -J mesa_pve70_munger
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 72:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=32G]"
#BSUB -oo /sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger/logs/%J.munger.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger/logs/%J.munger.err
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/sumstat_munger
export PATH="/sc/arion/work/sunh14/conda/envs/sumstat_munger/bin:$PATH"
hash -r
set -eo pipefail
cd /sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/1_DataPrep/Snakemake_38/Snakefile \
  --configfile /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/1_DataPrep/Snakemake_38/config_mesa_mono_v5_white_pve70_sensitivity_hg38.yaml \
  --profile lsf --jobs 300 --latency-wait 120 --printshellcmds \
  output/mesa_mono_v5_white.pve70_sensitivity.TensorQTL.cis_qtl_pairs.chrall.CPRA_b38.vcf.gz \
  output/mesa_mono_v5_white.pve70_sensitivity.TensorQTL.cis_qtl_pairs.chrall.CPRA_b38.vcf.gz.tbi \
  output/mesa_mono_v5_white.pve70_sensitivity.TensorQTL.cis_qtl_pairs.chrall.CPRA_b38.tsv.gz
LSF

## 3. Dependency-gated COSI production and no-force resume

After section 2 returns the munger job ID, replace `__MUNGER_JOB_ID__` below and execute this cell. LSF keeps COSI pending until the munger finishes successfully. The cell activates `cosi`, launches the sole Snakemake controller for all 126 pair endpoints, and uses the local Snakefile's per-unit SoS isolation. Rerun the same cell without force to resume.

In [ ]:
%%bash
mkdir -p /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/logs /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/runlog/snakemake_runtime/driver_full_prod/storage/local /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/runlog/snakemake_runtime/driver_full_prod/storage/remote
bsub <<'LSF'
#BSUB -J cosi_full18751
#BSUB -P acc_LOAD
#BSUB -q premium
#BSUB -W 144:00
#BSUB -n 1
#BSUB -R "span[hosts=1] rusage[mem=32G]"
#BSUB -w "done(__MUNGER_JOB_ID__)"
#BSUB -oo /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/logs/%J.cosi_driver.out
#BSUB -eo /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/logs/%J.cosi_driver.err
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/cosi
export PATH="/sc/arion/work/sunh14/conda/envs/cosi/bin:$PATH"
hash -r
set -eo pipefail
python -m snakemake \
  --snakefile /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/workflow/snakefile_combined.py \
  --configfile /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/config/full_legacy_core3_oldld_ad7_eqtl18.yaml \
  --directory /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun \
  --workflow-profile /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/workflow/profiles/lsf \
  --jobs 10000 --max-jobs-per-second 10000 --keep-going --latency-wait 120 --rerun-triggers mtime \
  --local-storage-prefix /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/runlog/snakemake_runtime/driver_full_prod/storage/local \
  --remote-job-local-storage-prefix /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/runlog/snakemake_runtime/driver_full_prod/storage/remote \
  --default-resources mem_mb=32768 runtime=720 lsf_project=acc_LOAD \
  --printshellcmds all
LSF

## 4. Status and completion

Replace the two retained job-ID tokens before executing. Scheduler completion is not scientific completion: inspect native accounting and logs, then verify the munger products, all 126 pair endpoints, and recorded weight failures.

In [ ]:
%%bash
set -o pipefail
MUNGER_JOB_ID=__MUNGER_JOB_ID__
COSI_PRODUCTION_JOB_ID=__COSI_PRODUCTION_JOB_ID__
bjobs -a -o 'jobid stat queue job_name exec_host exit_code' "$MUNGER_JOB_ID" "$COSI_PRODUCTION_JOB_ID" || true
bacct -l "$MUNGER_JOB_ID" "$COSI_PRODUCTION_JOB_ID"
find /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/results -type f \( -name '*.intact.results.tsv' -o -name '.no_coloc_hits*' \) -print | sort
find /sc/arion/projects/load/users/sunh14/myloid_paper/Myloid-paper/3_Production/COSI_pecotmr_rerun/hg38/full_legacy_core3_oldld_ad7_eqtl18/results -type f -name 'failed_genes.tsv' -size +0c -print | sort

## 5. Final validation and 5,095/8,080 aggregation

After production completes, require all 22 chromosome VCFs and indexes, the three combined products, exactly one valid terminal outcome for each of 126 pairs, and no unresolved workflow units. Then recompute the established 5,095/8,080 summaries and compare them with the legacy 525/707 sets using the canonical COSI result tables; do not create a separate controller or aggregation helper.

In [ ]:
%%bash
source /sc/arion/work/sunh14/conda/mambaforge/etc/profile.d/conda.sh
conda activate /sc/arion/work/sunh14/conda/envs/sumstat_munger
export PATH="/sc/arion/work/sunh14/conda/envs/sumstat_munger/bin:$PATH"
hash -r
set -eo pipefail
MUNGER_ROOT=/sc/arion/projects/load/users/sunh14/xqtl/MESA/work/route3_rna_pve70_sensitivity/mesa_mono_v5_white/posthoc/cosi_pecotmr_rerun/sumstats_munger
STEM=mesa_mono_v5_white.pve70_sensitivity.TensorQTL.cis_qtl_pairs
for CHR in {1..22}; do
  test -s "$MUNGER_ROOT/output/$STEM.chr$CHR.CPRA_b38.vcf.gz"
  test -s "$MUNGER_ROOT/output/$STEM.chr$CHR.CPRA_b38.vcf.gz.tbi"
done
test -s "$MUNGER_ROOT/output/$STEM.chrall.CPRA_b38.vcf.gz"
test -s "$MUNGER_ROOT/output/$STEM.chrall.CPRA_b38.vcf.gz.tbi"
test -s "$MUNGER_ROOT/output/$STEM.chrall.CPRA_b38.tsv.gz"
bcftools index -n "$MUNGER_ROOT/output/$STEM.chrall.CPRA_b38.vcf.gz"
printf 'munger_products=PASS chromosomes=22 combined=3\n'